# Kata 07 — Plan Mode (exploración segura)

> Spec: [`specs/007-plan-mode/spec.md`](../../specs/007-plan-mode/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
import json

client, settings = bootstrap(budget_calls=10)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Antes de modificar un repo desconocido, el agente trabaja en **read-only**: explora, escribe un plan en markdown, espera aprobación humana explícita, y sólo entonces se le habilitan herramientas de escritura.

En la API directa, "Plan Mode" se materializa controlando qué tools entrega el cliente: en fase 1 sólo herramientas de lectura (`list_files`, `read_file`); en fase 2, sólo después de la aprobación, también las de escritura (`apply_patch`).

## 2. Por qué importa

Un agente con permisos de escritura sobre un repo que no entiende es un dedo en el gatillo. Plan Mode separa "comprender" de "ejecutar"; el humano firma el plan, no cada edición individual.

## 3. Ejemplo correcto — dos fases con tools distintos

In [ ]:
FAKE_REPO = {
    "src/auth.py": "def login(): pass\n# TODO: validate credentials",
    "src/utils.py": "def hash(p): return p  # TODO: real hash",
    "README.md": "# proyecto demo",
}

def list_files(args):
    return list(FAKE_REPO.keys())

def read_file(args):
    return {"path": args["path"], "content": FAKE_REPO.get(args["path"], None)}

def apply_patch(args):
    FAKE_REPO[args["path"]] = args["new_content"]
    return {"path": args["path"], "status": "patched"}

LIST_TOOL = {"name": "list_files", "description": "Lista archivos del repo.", "input_schema": {"type":"object","properties":{},"required":[]}}
READ_TOOL = {"name": "read_file",  "description": "Lee un archivo.",         "input_schema": {"type":"object","properties":{"path":{"type":"string"}},"required":["path"]}}
PATCH_TOOL = {"name": "apply_patch","description": "Aplica un parche.",      "input_schema": {"type":"object","properties":{"path":{"type":"string"},"new_content":{"type":"string"}},"required":["path","new_content"]}}

READ_ONLY_TOOLS = [LIST_TOOL, READ_TOOL]
WRITE_TOOLS = [LIST_TOOL, READ_TOOL, PATCH_TOOL]
HANDLERS = {"list_files": list_files, "read_file": read_file, "apply_patch": apply_patch}

In [ ]:
def step(client, *, system, messages, tools):
    while True:
        resp = client.messages.create(system=system, messages=messages, tools=tools)
        if resp.stop_reason == "end_turn":
            return resp, messages
        if resp.stop_reason != "tool_use":
            return resp, messages
        tu = next(b for b in resp.content if b.type == "tool_use")
        if tu.name not in {t["name"] for t in tools}:
            # Tool fuera del set permitido: el bucle deniega.
            tool_result = {"error": "TOOL_NOT_ALLOWED_IN_THIS_MODE", "tool": tu.name}
        else:
            tool_result = HANDLERS[tu.name](tu.input)
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": [{
            "type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(tool_result),
        }]})

# FASE 1 — Plan Mode (read-only)
plan_system = (
    "Estás en PLAN MODE. Sólo tienes herramientas de lectura. "
    "Explora el repo y produce un plan en formato:\n"
    "PLAN:\n- archivo: cambio propuesto\n- ...\n"
    "Termina con el plan. NO intentes modificar nada."
)
messages = [{"role": "user", "content": "Hay TODOs en el código. Hazme un plan para resolverlos."}]
plan_resp, messages = step(client, system=plan_system, messages=messages, tools=READ_ONLY_TOOLS)
plan_text = next((b.text for b in plan_resp.content if b.type == "text"), "")
print("=== PLAN propuesto ===")
print(plan_text)

### 3.1 Aprobación humana explícita

In [ ]:
approval = input("¿Aprobar el plan? [y/N]: ").strip().lower() if False else "y"  # auto-aprobado para que corra
print("aprobación:", approval)
assert approval == "y", "Plan no aprobado; abortamos antes de fase 2."


### 3.2 FASE 2 — Direct Execution (write habilitado)

In [ ]:
exec_system = (
    "El plan fue aprobado. Ahora puedes usar apply_patch para implementar EXACTAMENTE el plan. "
    "No te desvíes; no agregues cambios fuera del plan."
)
# Continuamos la sesión: el modelo recordará el plan que produjo
messages.append({"role": "user", "content": "Plan aprobado. Implementa exactamente lo planeado."})
exec_resp, messages = step(client, system=exec_system, messages=messages, tools=WRITE_TOOLS)
print("=== final ===")
print(next((b.text for b in exec_resp.content if b.type == "text"), "")[:600])
print("\nrepo después:")
for k, v in FAKE_REPO.items():
    print(f"  {k}: {v[:60]}")

## 4. Anti-patrón — escritura desde el minuto cero

In [ ]:
# Resetear repo
FAKE_REPO_RESET = {
    "src/auth.py": "def login(): pass\n# TODO: validate credentials",
    "src/utils.py": "def hash(p): return p  # TODO: real hash",
    "README.md": "# proyecto demo",
}
FAKE_REPO.clear(); FAKE_REPO.update(FAKE_REPO_RESET)

bad_system = "Eres un agente refactor. Resuelve los TODOs del repo."
messages = [{"role": "user", "content": "Hay TODOs en el código. Resuélvelos."}]
bad_resp, _ = step(client, system=bad_system, messages=messages, tools=WRITE_TOOLS)
print(next((b.text for b in bad_resp.content if b.type == "text"), "")[:400])
print("\nrepo después:")
for k, v in FAKE_REPO.items():
    print(f"  {k}: {v[:80]}")

Sin Plan Mode, el agente puede escribir parches a archivos que no entiende, sin que un humano haya visto el plan completo. En un repo real, la primera edición en mal lugar genera un incidente.

## 5. Argumento de certificación

- **Dos modos con permisos distintos**: el cliente entrega un set de tools en cada fase. La transición es explícita y registrada.
- **Plan Mode no es opt-in del modelo, es opt-in del cliente.** Aunque el modelo "sepa" que existe `apply_patch`, no puede invocarla porque no está en `tools=`.
- **Aprobación humana sobre el plan, no sobre cada edición.** Reduce fricción para el humano y eleva el costo de errores no planificados.
- **Conexión con otros katas**: el set de tools en fase 1 es un caso particular de Kata 2 (gating); el plan en markdown es un scratchpad estructurado (Kata 18).

## 6. Auto-evaluación

**1. ¿Qué pasa si durante la ejecución el agente descubre que el plan era incorrecto?**

Debería detenerse y volver a fase de plan. En esta implementación lo modelaría devolviendo un `tool_result` con `error_code: PLAN_INVALIDATED` y un nuevo turno donde el cliente regresa a Plan Mode (read-only) hasta que se apruebe un plan revisado.

**2. ¿Cómo aseguro que Plan Mode bloquea TODA escritura, no sólo las "obvias"?**

El cliente pasa explícitamente `tools=READ_ONLY_TOOLS`. No hay forma de que el modelo invoque `apply_patch`: el SDK rechaza tool_use con nombres que no están en la declaración. Adicionalmente, el bucle valida `tu.name in {t["name"] for t in tools}` antes de ejecutar.

**3. ¿Dónde se almacena la aprobación humana para que la auditoría la encuentre?**

En el log de eventos del bucle, con el plan markdown como adjunto. En producción, en una tabla `plan_approvals(plan_hash, approver, timestamp)` y el agente en fase 2 verifica que `hash(plan)` coincide con uno aprobado.